## 🔐 Prerequisites

Before running the first cell, make sure you're authenticated with Azure CLI. Run this command in your terminal:

```bash
az login
```

or

```bash
az login --use-device-code
```

# 💬 Azure AI Agent with Thread Management - Multi-Turn Conversations 🏦

This notebook demonstrates working with conversation threads using `AzureAIProjectAgentProvider` for multi-turn conversations like loan application discussions and financial planning sessions.

## Features Covered:
- Creating and managing persistent conversation threads
- Using `agent.get_new_thread()` for thread management
- Thread lifecycle management (create, use, delete)
- Multi-turn conversation continuity for banking discussions
- Thread initialization and validation with `store=False` option

### ⚠️ Important Note ⚠️
> **Threads maintain conversation context across multiple interactions. This is essential for complex banking discussions like loan applications or investment planning.**

## Prerequisites

Before running this notebook, ensure you have:

1. **Azure AI Project**: Access to an Microsoft Foundry project with deployed models
2. **Authentication**: Azure CLI installed and authenticated (`az login --use-device-code`)
3. **Environment Variables**: Set up your `.env` file with:
   - `AI_FOUNDRY_PROJECT_ENDPOINT`
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME`
4. **Dependencies**: Required agent-framework packages installed

If you need to use a different tenant:
```bash
az login --tenant <tenant-id>
```

This example demonstrates how to work with existing threads to maintain conversation continuity in banking scenarios.

## Import Libraries

Import the required libraries using the `AzureAIAgentsProvider` API:

In [1]:
import os
from pathlib import Path
from random import randint, uniform
from typing import Annotated

from agent_framework import AgentSession
from agent_framework_foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from pydantic import Field
from dotenv import load_dotenv

# Load environment variables
load_dotenv('../../../.env')

# Verify environment setup
endpoint = os.getenv('AI_FOUNDRY_PROJECT_ENDPOINT')
model = os.getenv('AZURE_AI_MODEL_DEPLOYMENT_NAME')

print("🔧 Environment Configuration:")
print(f"✅ Project Endpoint: {endpoint[:50]}..." if endpoint else "❌ AI_FOUNDRY_PROJECT_ENDPOINT not set")
print(f"✅ Model Deployment: {model}" if model else "❌ AZURE_AI_MODEL_DEPLOYMENT_NAME not set")

<frozen abc>:106: ExperimentalWarning: [HARNESS] MemoryStore is experimental and may change or be removed in future versions without notice.
<frozen abc>:106: ExperimentalWarning: [SKILLS] SkillResource is experimental and may change or be removed in future versions without notice.


🔧 Environment Configuration:
✅ Project Endpoint: https://kd-foundry.services.ai.azure.com/api/proje...
✅ Model Deployment: gpt-4o


## Check Environment Variables

Verify that the required environment variables are set:

In [2]:
# Check required environment variables
endpoint = os.getenv('AI_FOUNDRY_PROJECT_ENDPOINT')
if endpoint:
    print(f"✅ AI_FOUNDRY_PROJECT_ENDPOINT: {endpoint}")
else:
    print("❌ AI_FOUNDRY_PROJECT_ENDPOINT: Not set")
    print("⚠️  Please set the AI_FOUNDRY_PROJECT_ENDPOINT environment variable")

✅ AI_FOUNDRY_PROJECT_ENDPOINT: https://kd-foundry.services.ai.azure.com/api/projects/proj-kd


## Define Function Tools 🏦

Let's define banking functions for our loan application discussions:

In [3]:
def get_loan_prequalification(
    loan_amount: Annotated[float, Field(description="Requested loan amount in dollars.")],
    credit_score: Annotated[int, Field(description="Customer's credit score.")],
    annual_income: Annotated[float, Field(description="Customer's annual income in dollars.")],
) -> str:
    """Check loan prequalification status based on customer data."""
    # Simple prequalification logic for demo
    dti = loan_amount / (annual_income * 30)  # Simplified debt-to-income estimate
    
    if credit_score >= 700 and dti < 0.43:
        status = "Pre-Qualified ✅"
        rate = "5.99% - 6.75%"
    elif credit_score >= 650 and dti < 0.50:
        status = "Conditionally Pre-Qualified ⚠️"
        rate = "7.25% - 8.50%"
    else:
        status = "Additional Review Required 📋"
        rate = "8.50% - 12.00%"
    
    return f"""Loan Pre-Qualification Result:
    Requested Amount: ${loan_amount:,.2f}
    Credit Score: {credit_score}
    Annual Income: ${annual_income:,.2f}
    Status: {status}
    Estimated Rate Range: {rate}"""

def get_required_documents(
    loan_type: Annotated[str, Field(description="Type of loan: mortgage, auto, personal")],
) -> str:
    """Get list of required documents for loan application."""
    docs = {
        "mortgage": [
            "Government-issued ID",
            "Pay stubs (last 30 days)",
            "W-2 forms (last 2 years)",
            "Tax returns (last 2 years)",
            "Bank statements (last 2 months)",
            "Proof of assets",
            "Property information"
        ],
        "auto": [
            "Government-issued ID",
            "Proof of income",
            "Proof of insurance",
            "Vehicle information (VIN, mileage)"
        ],
        "personal": [
            "Government-issued ID",
            "Proof of income",
            "Proof of residence"
        ]
    }
    loan_type = loan_type.lower()
    if loan_type in docs:
        doc_list = "\n    - ".join(docs[loan_type])
        return f"Required Documents for {loan_type.title()} Loan:\n    - {doc_list}"
    return "Please specify: mortgage, auto, or personal loan"

## Create and Use Existing Thread 💬

This example shows how to:
1. Create a thread that persists for the conversation
2. Use `agent.get_new_thread(service_thread_id=...)` to continue conversations
3. Properly clean up thread resources

**Key API Methods**: 
- `agent.get_new_thread()` - Creates new thread (with optional `store=False`)
- `agent.get_new_thread(service_thread_id=...)` - Uses existing thread

In [4]:
async def main() -> None:
    print("=== \U0001f4ac Azure AI Agent with Session Management - Loan Application ===")

    credential = AzureCliCredential()
    client = FoundryChatClient(credential=credential)
    
    agent = client.as_agent(
        name="LoanAdvisorAgent",
        instructions="""You are a senior Loan Advisor at a retail bank.
        You help customers through their loan application process.
        Be professional, thorough, and maintain context across the conversation.""",
        tools=[get_loan_prequalification, get_required_documents],
    )
    
    async with agent:
        session = agent.create_session()
        print(f"\u2705 Session created: {session.session_id}")
        
        # Turn 1: Customer introduces themselves
        query1 = "Hi, I'm interested in getting a home loan. Can you check if I'd prequalify for $350,000 with a credit score of 720 and annual income of $95,000?"
        print(f"\n\U0001f914 Customer: {query1}")
        response1 = await agent.run(query1, session=session)
        print(f"\U0001f3e6 Advisor: {response1.text}")
        
        # Turn 2: Follow-up question (maintains context from turn 1)
        query2 = "Based on that, what documents would I need for a mortgage?"
        print(f"\n\U0001f914 Customer: {query2}")
        response2 = await agent.run(query2, session=session)
        print(f"\U0001f3e6 Advisor: {response2.text}")
        
        print("\n\u2705 Multi-turn session completed!")

## 🚀 Execute the Initial Loan Discussion

In [5]:
await main()

=== 💬 Azure AI Agent with Session Management - Loan Application ===


✅ Session created: ecb2a8c4-19d2-4cff-bb74-923c245433a4

🤔 Customer: Hi, I'm interested in getting a home loan. Can you check if I'd prequalify for $350,000 with a credit score of 720 and annual income of $95,000?


🏦 Advisor: You've been pre-qualified for a home loan of $350,000. 

- **Estimated Rate Range**: 5.99% - 6.75%
- **Credit Score**: 720
- **Annual Income**: $95,000

Would you like to proceed with the application or need details on the required documents?

🤔 Customer: Based on that, what documents would I need for a mortgage?


🏦 Advisor: For a mortgage loan, you'll need the following documents:

- Government-issued ID
- Pay stubs from the last 30 days
- W-2 forms for the last 2 years
- Tax returns for the last 2 years
- Bank statements from the last 2 months
- Proof of assets
- Property information

Feel free to ask if you need more information or assistance with the application process!

✅ Multi-turn session completed!


## 🔄 Multi-Turn Loan Application Flow with Thread Continuity

In [6]:
async def multi_turn_loan_application() -> None:
    """
    Demonstrates a complete multi-turn loan application conversation
    where session context maintains the discussion history.
    """
    print("=== \U0001f4ac Multi-Turn Loan Application Workflow ===")
    
    credential = AzureCliCredential()
    client = FoundryChatClient(credential=credential)
    
    advisor = client.as_agent(
        name="LoanApplicationAdvisor",
        instructions="""You are a comprehensive Loan Application Advisor. Guide customers through:
        1. Understanding loan options
        2. Prequalification assessment  
        3. Document requirements
        4. Next steps in the application process
        
        Remember the conversation context and reference earlier information when helpful.
        Be encouraging while being realistic about qualifications.""",
        tools=[get_loan_prequalification, get_required_documents],
    )
    
    async with advisor:
        session = advisor.create_session()
        
        # Loan application conversation flow
        conversation = [
            "Hi, I'm looking to buy my first home and need information about mortgage options.",
            "Can you check if I'd prequalify? My credit score is 680, annual income is $72,000, and I'm looking at homes around $280,000.",
            "Based on my prequalification status, what documents would I need for a conventional mortgage?",
            "Given my situation, would you recommend I improve my credit score first before applying?"
        ]
        
        for i, message in enumerate(conversation, 1):
            print(f"\n\U0001f4e9 Turn {i} - Customer: {message}")
            response = await advisor.run(message, session=session)
            print(f"\U0001f3e6 Advisor: {response.text}")
            print("-" * 50)
        
        print("\n\u2705 Loan application session completed")

## 🚀 Execute Multi-Turn Loan Application

In [7]:
await multi_turn_loan_application()

=== 💬 Multi-Turn Loan Application Workflow ===



📩 Turn 1 - Customer: Hi, I'm looking to buy my first home and need information about mortgage options.


🏦 Advisor: Congratulations on considering buying your first home! Let's explore some mortgage options and how they might fit your needs.

### Common Mortgage Types:

1. **Fixed-Rate Mortgage**:
   - Interest rate remains the same throughout the loan term, providing predictability in monthly payments.
   - Typically available in 15, 20, or 30-year terms.

2. **Adjustable-Rate Mortgage (ARM)**:
   - Interest rates can change periodically based on market conditions, usually offering a lower initial rate.
   - Common terms include 5/1, 7/1, or 10/1 ARM, where the first number indicates years of fixed rates, and the second is the adjustment frequency.

3. **FHA Loans**:
   - Insured by the Federal Housing Administration, ideal for first-time homebuyers with lower credit scores.
   - Requires a lower down payment compared to conventional loans.

4. **VA Loans**:
   - Available to veterans and active-duty military members.
   - Offers favorable terms without requiring a down payment or mortga

🏦 Advisor: You are **conditionally pre-qualified** for a mortgage loan of $280,000. Here are the details:

- **Credit Score**: 680
- **Annual Income**: $72,000
- **Estimated Interest Rate Range**: 7.25% - 8.50%

### What Does "Conditionally Pre-Qualified" Mean?

This status indicates that, based on the information provided, you meet certain criteria for loan approval. However, it's subject to further verification and meeting specific lender requirements.

### Next Steps:

1. **Verify Documentation**: Ensure you have documents ready for income, assets, and identity verification.
2. **Improve Your Credit Score**: If possible, work on enhancing your credit score to potentially secure better rates.
3. **Shop Around**: Consider different lenders to find competitive rates and terms.
4. **Consult a Mortgage Advisor**: They can provide personalized advice and explore options fitting your needs.

### Required Document Checklist:

Would you like me to provide a list of required documents for a m

🏦 Advisor: For a conventional mortgage, you'll need the following documents:

1. **Government-issued ID**: Such as a driver's license or passport.
2. **Pay Stubs**: Covering the last 30 days.
3. **W-2 Forms**: From the last 2 years.
4. **Tax Returns**: Complete returns for the last 2 years.
5. **Bank Statements**: Statements from the last 2 months.
6. **Proof of Assets**: Documentation of savings, investments, or other assets.
7. **Property Information**: Details about the home you intend to purchase.

### Next Steps:

- **Gather Documentation**: Start organizing these documents.
- **Consult with Lenders**: Discuss your options, rates, and terms with different lenders.
- **Consider Pre-Approval**: Getting pre-approved can strengthen your offer when you're ready to make a bid on a home.

Feel free to ask any more questions or if you need further assistance with the process!
--------------------------------------------------

📩 Turn 4 - Customer: Given my situation, would you recommend I

🏦 Advisor: Improving your credit score could potentially offer several benefits:

### Benefits of Improving Your Credit Score:

1. **Lower Interest Rates**: Better scores often qualify for lower rates, reducing lifetime loan costs.
2. **Increased Approval Likelihood**: Enhances your attractiveness to lenders.
3. **Better Loan Terms**: Possibility of more favorable loan conditions.

### Steps to Improve Your Credit Score:

- **Pay Bills on Time**: Maintain consistent, timely payments.
- **Reduce Debt**: Lower your credit card balances and other debt.
- **Avoid New Credit**: Minimize new credit inquiries before applying.
- **Check Reports**: Review your credit report for errors and have any inaccuracies corrected.

### Consider Your Timeline:

If you have flexibility and can wait a bit before purchasing a home, improving your score could be beneficial. However, if you're in a competitive market or eager to buy soon, you can proceed with the current score while possibly working on gradual

## 📝 Key Takeaways

### Thread Management in Applications

1. **Thread Persistence**: Threads maintain conversation context across multiple interactions, essential for complex multi-turn discussions

2. **Creating Threads**: Use `AgentsClient` to create threads before agent interactions:
   ```python
   created_thread = await agents_client.threads.create()
   ```

3. **Using Existing Threads**: Connect agent to existing thread for conversation continuity:
   ```python
   thread = agent.get_new_thread(service_thread_id=created_thread.id)
   ```

4. **Thread Lifecycle**: Always clean up threads when conversation is complete:
   ```python
   await agents_client.threads.delete(thread_id)
   ```

### Use Case Benefits

- **Compliance**: Persistent threads create audit trails for regulatory compliance
- **Customer Experience**: Customers don't need to repeat information
- **Context-Aware Responses**: Agents can reference earlier parts of the conversation
- **Multi-Step Processes**: Perfect for loan applications, account onboarding, and consultations

### ⚠️ Disclaimer
This example uses simulated data for demonstration purposes. In production:
- Connect to actual backend systems
- Implement proper authentication and authorization
- Follow all regulatory requirements
- Ensure compliance for sensitive data